# Extreme Climate Event Prediction (Starter Workflow)\n\nThis notebook is a **reproducible baseline** for rare-event classification ("extremes"). It uses **synthetic** data to demonstrate best-practice evaluation (PR-AUC, calibration, leakage awareness).\n\nReplace the synthetic generator with your own climate dataset when ready.

## 1) Imports

In [ ]:
import numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import (\n    average_precision_score,\n    precision_recall_curve,\n    roc_auc_score,\n    brier_score_loss,\n    classification_report,\n)\nfrom sklearn.calibration import calibration_curve\n\nsns.set_theme(style="whitegrid")\nRNG = np.random.default_rng(42)

## 2) Create a synthetic "daily climate" dataset\n\nIn a real workflow, you would load a table where each row is a day (or timestep) with covariates and a target.

In [ ]:
def make_synthetic_climate_table(n=5000, rng=RNG):\n    t = np.arange(n)\n    day_of_year = t % 365\n    seasonal = np.sin(2 * np.pi * day_of_year / 365.0)\n\n    temp = 15 + 10 * seasonal + rng.normal(0, 2.0, size=n)\n    humidity = 60 - 15 * seasonal + rng.normal(0, 5.0, size=n)\n    pressure = 1013 + rng.normal(0, 8.0, size=n)\n    wind = np.clip(rng.gamma(shape=2.0, scale=1.5, size=n), 0, None)\n\n    base = 1.0 + 0.3 * (humidity / 100.0) + 0.1 * wind\n    spikes = rng.binomial(1, 0.06, size=n) * rng.lognormal(mean=1.8, sigma=0.6, size=n)\n    target = np.clip(base + spikes + rng.normal(0, 0.2, size=n), 0, None)\n\n    return pd.DataFrame({\n        "t": t,\n        "day_of_year": day_of_year,\n        "temp_c": temp,\n        "humidity_pct": humidity,\n        "pressure_hpa": pressure,\n        "wind_ms": wind,\n        "target": target,\n    })\n\ndf = make_synthetic_climate_table()\ndf.head()

## 3) Define the "extreme event" label\n\nWe define extremes using a percentile threshold (e.g., 95th percentile). This is common for impacts-focused definitions but should be adapted to domain needs.

In [ ]:
extreme_quantile = 0.95\nthreshold = float(df["target"].quantile(extreme_quantile))\ndf["is_extreme"] = (df["target"] >= threshold).astype(int)\n\ndf["is_extreme"].value_counts(normalize=True)

In [ ]:
plt.figure(figsize=(7,4))\nsns.histplot(df["target"], bins=50)\nplt.axvline(threshold, color="red", linestyle="--", linewidth=2, label=f"{extreme_quantile:.0%} threshold")\nplt.title("Synthetic target distribution")\nplt.legend()\nplt.tight_layout()\nplt.show()

## 4) Train/test split\n\nFor real climate problems, prefer **time-aware** or **space-aware** splits to avoid leakage. For this synthetic demo, we use a stratified split.

In [ ]:
feature_cols = ["day_of_year", "temp_c", "humidity_pct", "pressure_hpa", "wind_ms"]\nX = df[feature_cols]\ny = df["is_extreme"].to_numpy()\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.25, random_state=42, stratify=y\n)\nX_train.shape, X_test.shape

## 5) Build models and evaluate with rare-event metrics\n\nWe report **PR-AUC** (average precision) and **Brier score** (calibration). ROC-AUC is included but can look deceptively strong under severe imbalance.

In [ ]:
pre = ColumnTransformer(\n    transformers=[(\n        "num",\n        Pipeline([\n            ("imputer", SimpleImputer(strategy="median")),\n            ("scaler", StandardScaler()),\n        ]),\n        feature_cols,\n    )],\n    remainder="drop",\n)\n\nmodels = {\n    "logreg": Pipeline([\n        ("pre", pre),\n        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),\n    ]),\n    "rf": Pipeline([\n        ("pre", pre),\n        ("clf", RandomForestClassifier(\n            n_estimators=400,\n            random_state=42,\n            class_weight="balanced_subsample",\n            n_jobs=-1,\n        )),\n    ]),\n}\n\nrows = []\nfor name, model in models.items():\n    model.fit(X_train, y_train)\n    prob = model.predict_proba(X_test)[:, 1]\n    pred = (prob >= 0.5).astype(int)\n\n    rows.append({\n        "model": name,\n        "roc_auc": roc_auc_score(y_test, prob),\n        "pr_auc": average_precision_score(y_test, prob),\n        "brier": brier_score_loss(y_test, prob),\n    })\n    print("\n===", name, "===")\n    print(classification_report(y_test, pred, zero_division=0))\n\nmetrics = pd.DataFrame(rows).sort_values("pr_auc", ascending=False)\nmetrics

## 6) Precision–Recall curves and calibration curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))\n\nfor name, model in models.items():\n    prob = model.predict_proba(X_test)[:, 1]\n    precision, recall, _ = precision_recall_curve(y_test, prob)\n    ap = average_precision_score(y_test, prob)\n    axes[0].plot(recall, precision, label=f"{name} (AP={ap:.3f})")\n\naxes[0].set_title("Precision–Recall")\naxes[0].set_xlabel("Recall")\naxes[0].set_ylabel("Precision")\naxes[0].legend()\n\nfor name, model in models.items():\n    prob = model.predict_proba(X_test)[:, 1]\n    frac_pos, mean_pred = calibration_curve(y_test, prob, n_bins=10, strategy="quantile")\n    axes[1].plot(mean_pred, frac_pos, marker="o", label=name)\n\naxes[1].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)\naxes[1].set_title("Calibration")\naxes[1].set_xlabel("Mean predicted probability")\naxes[1].set_ylabel("Fraction of positives")\naxes[1].legend()\n\nplt.tight_layout()\nplt.show()

## Next steps for a real dataset\n\n- Use **blocked time splits** (e.g., train on earlier years, test on later years).\n- Consider **spatial CV** if you want generalization to unseen locations.\n- Add features: lags, rolling statistics, climate indices, reanalysis variables.\n- Compare modern methods: gradient boosting, temporal CNN/RNN/Transformers (with careful validation).\n- Calibrate probabilities (Platt scaling / isotonic) for warning systems.